# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset contains detailed clinicopathological and molecular variables for 77 cancer survivors with second primary colorectal cancer.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a brief description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id, name, and description
print('Available Record Sets:')
for recordset in dataset.record_sets:
    print(f"- @id: {recordset.id} | name: {getattr(recordset, 'name', 'N/A')} | description: {getattr(recordset, 'description', 'No description')}")

# For each record set, list its fields by @id and name
for recordset in dataset.record_sets:
    print(f"\nFields in Record Set @id={recordset.id}:")
    for field in recordset.fields:
        print(f"  - @id: {field.id} | name: {getattr(field, 'name', 'N/A')} | dataType: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}

record_set_ids = [rs.id for rs in dataset.record_sets]
print("\nExtracting the following record sets:")
for rsid in record_set_ids:
    print(f"- {rsid}")
    # Retrieve records (list of dicts)
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

# For demonstration: print columns for the first record set loaded
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"\nFields (@id) in main DataFrame ({main_record_set}):")
    print(list(dataframes[main_record_set].columns))
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Use field `@id`s to reference variables.

In [ ]:
# For this EDA, pick a numeric field and a grouping field by their @id
# Let's display all numeric-typed field @ids for main_record_set
main_df = dataframes[main_record_set]

print("\nNumeric candidate columns (by @id) in the main record set:")
numeric_fields = []
group_fields = []
for f in dataset.record(main_record_set).fields:
    if getattr(f, 'data_type', '').lower() in ['integer', 'float', 'number']:
        numeric_fields.append(f.id)
    if getattr(f, 'data_type', '').lower() in ['text', 'string', 'categorical']:
        group_fields.append(f.id)
print(numeric_fields)
print("Grouping field candidates (by @id):")
print(group_fields)

# For exploration, pick the first available numeric and group field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = main_df.select_dtypes(include=['number']).columns[0]

if group_fields:
    group_field_id = group_fields[0]
    print(f"Using group field: {group_field_id}")
else:
    group_field_id = main_df.columns[0]

# Drop NA values for selected columns
filtered_df = main_df.dropna(subset=[numeric_field_id])
filtered_df = filtered_df[filtered_df[numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')).notnull()]
filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)

threshold = filtered_df[numeric_field_id].median()  # Use median for threshold
filtered_df2 = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
print(filtered_df2[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df2[f"{numeric_field_id}_normalized"] = (filtered_df2[numeric_field_id] - filtered_df2[numeric_field_id].mean()) / filtered_df2[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df2[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the selected group field (if it exists)
if group_field_id in filtered_df2.columns:
    grouped_df = filtered_df2.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset by referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df2[numeric_field_id], bins=10, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group (if possible)
if group_field_id in filtered_df2.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=filtered_df2[group_field_id], y=filtered_df2[numeric_field_id])
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR² clinical dataset using `mlcroissant`. All dataset entities were referenced by their unique `@id` and data was dynamically processed. For further statistical or machine learning tasks, continue using the DataFrames generated and reference data elements by their IDs for transparent and reproducible science.